<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 16 - Способы оплаты (PaymentMethod)


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

В проекте создан базовый класс `PaymentMethod` (ID, название, мин. сумма) с методами `ProcessPayment`, `CheckMinimumAmount` и `GetPaymentDetails`. От него наследуются 3 класса: `OnlinePayment` (URL платёжной системы), `BankTransfer` (банковские данные, мин. сумма с учётом комиссии банка) и `CashPayment` (место выдачи наличных). В производных классах переопределены методы базового класса, за счёт этого показан полиморфизм.

#### Дополнительное задание
Добавлены новые атрибуты и методы в базовый и производные классы. Явная реализация интерфейса сделана для `IReceipt` (метод `GetReceipt`), а управление зависимостями — в классе `PaymentService`, который получает `PaymentMethod` и `IPaymentLogger` через конструктор.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:
// интерфейс для чека (реализ. явно)
public interface IReceipt
{
    string GetReceipt(decimal amount);
}

// интерфейс для логов (нужен для DI)
public interface IPaymentLogger
{
    void Log(string message);
}

public class ConsoleLogger : IPaymentLogger
{
    public void Log(string message)
    {
        Console.WriteLine("[ЛОГ] " + message);
    }
}

// базовый класс
public class PaymentMethod : IReceipt
{
    public int PaymentMethodId { get; set; }
    public string MethodName { get; set; }
    public decimal MinAmount { get; set; }

    // доп. атрибуты
    public decimal CommissionPercent { get; set; }
    public int PaymentsCount { get; set; }
    public bool IsActive { get; set; } = true;

    public virtual bool ProcessPayment(decimal amount)
    {
        if (!IsActive)
        {
            Console.WriteLine($"{MethodName} сейчас отключен.");
            return false;
        }
        if (!CheckMinimumAmount(amount))
        {
            Console.WriteLine($"Сумма {amount} меньше мин. суммы ({MinAmount}).");
            return false;
        }
        PaymentsCount++;
        Console.WriteLine($"Платёж {amount} прошёл через {MethodName}.");
        return true;
    }

    public virtual bool CheckMinimumAmount(decimal amount)
    {
        return amount >= MinAmount;
    }

    public virtual string GetPaymentDetails()
    {
        return $"ID: {PaymentMethodId}, способ: {MethodName}, мин. сумма: {MinAmount}";
    }

    // доп. методы
    public decimal CalculateCommission(decimal amount)
    {
        return amount * CommissionPercent / 100;
    }

    public void Deactivate()
    {
        IsActive = false;
    }

    // явная реализ. интерфейса (видна только через IReceipt)
    string IReceipt.GetReceipt(decimal amount)
    {
        return $"Чек: {MethodName}, сумма {amount}, комиссия {CalculateCommission(amount)}";
    }
}

// онлайн оплата
public class OnlinePayment : PaymentMethod
{
    public string PaymentUrl { get; set; }
    public string Currency { get; set; }

    public override bool ProcessPayment(decimal amount)
    {
        Console.WriteLine($"Переходим на {PaymentUrl} для оплаты...");
        return base.ProcessPayment(amount);
    }

    public void SendConfirmation()
    {
        Console.WriteLine($"Подтверждение отправлено ({Currency}).");
    }
}

// банковский перевод
public class BankTransfer : PaymentMethod
{
    public string BankData { get; set; }
    public decimal BankFee { get; set; }

    // мин. сумма с учётом комиссии банка
    public override bool CheckMinimumAmount(decimal amount)
    {
        return amount - BankFee >= MinAmount;
    }

    public override string GetPaymentDetails()
    {
        return base.GetPaymentDetails() + $", реквизиты: {BankData}, комиссия банка: {BankFee}";
    }

    public void CheckBankData()
    {
        Console.WriteLine($"Реквизиты {BankData} проверены.");
    }
}

// наличные
public class CashPayment : PaymentMethod
{
    public string CashPickupPoint { get; set; }
    public int DaysToPickup { get; set; }

    public override string GetPaymentDetails()
    {
        return base.GetPaymentDetails() + $", место выдачи: {CashPickupPoint}";
    }

    public void GiveCash(decimal amount)
    {
        Console.WriteLine($"Выдано {amount} в {CashPickupPoint}, забрать за {DaysToPickup} дн.");
    }
}

// сервис берёт способ оплаты и логгер через конструктор (DI)
public class PaymentService
{
    private readonly PaymentMethod _method;
    private readonly IPaymentLogger _logger;

    public PaymentService(PaymentMethod method, IPaymentLogger logger)
    {
        _method = method;
        _logger = logger;
    }

    public void Pay(decimal amount)
    {
        _logger.Log($"Начало оплаты: {_method.MethodName}, {amount}");
        bool ok = _method.ProcessPayment(amount);
        if (ok)
        {
            IReceipt receipt = _method; // чек только через интерфейс
            Console.WriteLine(receipt.GetReceipt(amount));
        }
        _logger.Log(ok ? "Оплата успешна" : "Оплата не прошла");
    }
}

// проверка
IPaymentLogger logger = new ConsoleLogger();

var online = new OnlinePayment
{
    PaymentMethodId = 1, MethodName = "Онлайн оплата", MinAmount = 10,
    CommissionPercent = 2, PaymentUrl = "https://pay.example.com", Currency = "RUB"
};

var bank = new BankTransfer
{
    PaymentMethodId = 2, MethodName = "Банк. перевод", MinAmount = 100,
    CommissionPercent = 1, BankData = "40817810000000000001", BankFee = 20
};

var cash = new CashPayment
{
    PaymentMethodId = 3, MethodName = "Наличные", MinAmount = 50,
    CommissionPercent = 0, CashPickupPoint = "Касса №1", DaysToPickup = 3
};

new PaymentService(online, logger).Pay(500);
Console.WriteLine();
new PaymentService(bank, logger).Pay(110);  // 110 - 20 = 90 < 100, не пройдёт
Console.WriteLine();
new PaymentService(bank, logger).Pay(200);
Console.WriteLine();
new PaymentService(cash, logger).Pay(300);
Console.WriteLine();

// полиморфизм: один список, разное поведение
List<PaymentMethod> methods = new List<PaymentMethod> { online, bank, cash };
foreach (var m in methods)
{
    Console.WriteLine(m.GetPaymentDetails());
}

online.SendConfirmation();
bank.CheckBankData();
cash.GiveCash(300);

The below script needs to be able to find the current output cell; this is an easy method to get it.

[ЛОГ] Начало оплаты: Онлайн оплата, 500
Переходим на https://pay.example.com для оплаты...
Платёж 500 прошёл через Онлайн оплата.
Чек: Онлайн оплата, сумма 500, комиссия 10
[ЛОГ] Оплата успешна

[ЛОГ] Начало оплаты: Банк. перевод, 110
Сумма 110 меньше мин. суммы (100).
[ЛОГ] Оплата не прошла

[ЛОГ] Начало оплаты: Банк. перевод, 200
Платёж 200 прошёл через Банк. перевод.
Чек: Банк. перевод, сумма 200, комиссия 2
[ЛОГ] Оплата успешна

[ЛОГ] Начало оплаты: Наличные, 300
Платёж 300 прошёл через Наличные.
Чек: Наличные, сумма 300, комиссия 0
[ЛОГ] Оплата успешна

ID: 1, способ: Онлайн оплата, мин. сумма: 10
ID: 2, способ: Банк. перевод, мин. сумма: 100, реквизиты: 40817810000000000001, комиссия банка: 20
ID: 3, способ: Наличные, мин. сумма: 50, место выдачи: Касса №1
Подтверждение отправлено (RUB).
Реквизиты 40817810000000000001 проверены.
Выдано 300 в Касса №1, забрать за 3 дн.
